In [5]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, make_scorer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
import time

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, RobustScaler, PowerTransformer, OneHotEncoder, PolynomialFeatures, FunctionTransformer, QuantileTransformer



from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#from tensorflow.keras.utils import plot_model

#import plotly.graph_objects as go
import plotly.express as px
#import plotly.io as pio
import duckdb
from sklearn.model_selection import train_test_split
import math
#import warnings

import tensorflow as tf


#pio.templates.default = "seaborn"

#pio.templates["custom_theme"] = go.layout.Template(
#    layout=go.Layout(
#        #paper_bgcolor='rgba(0,0,0,0)', 
#        plot_bgcolor="rgba(230,230,230,255)",
#
#        colorway=px.colors.qualitative.D3
#
#    )
#)

#pio.templates.default = 'custom_theme'

lab_dict = {
        "udi":"UDI",
        "product_id":"Product ID",
        "type":"Type",
        "air_temperature_k":"Air temperature [K]",
        'process_temperature_k':'Process temperature [K]',
        'rotational_speed_rpm':'Rotational speed [RPM]',
        'torque_nm':'Torque [Nm]',
        'tool_wear_min':'Tool wear [min]',
        "machine_failure":"Machine failure",
        "twf":"Tool Wear Failure",
        "hdf":"Heat Dissipation  Failure",
        "pwf":"Power Failure",
        "osf":"Over Strain Failure",
        "rnf":"Random Failure",
        "delta_temp_k": "Temperature Difference [K]",
        "power_w":"Power [kW]",
        "cons_number":"#Product",
        "wear_torque":"Tool Wear [min] x Torque [Nm] "
        }

In [ ]:
def load_data(incl_cols, target):

    with duckdb.connect("data/team_data.duckdb") as conn:
        df = conn.execute("SELECT * FROM ai4i2020").fetchdf()

    df_features = df
    df_features.columns = df_features.columns.str.lower().str.replace(" ","_").str.replace("[","").str.replace("]","")

    

    # generate additional columns
    # power measured in W = 1Nm/s // torque in Nm // rotational speed in rpm => * 60 (sec) / 2pi (one ration)
    df_features["power_w"] = (df_features["torque_nm"] / (df_features["rotational_speed_rpm"] * 60 / (2 * math.pi))) * 1000
    df_features["delta_temp_k"] = df_features["process_temperature_k"]-df_features["air_temperature_k"]
    df_features["wear_torque"]  = df_features["tool_wear_min"]*df_features["torque_nm"]
    
    # set categorical columns
    target_cols =[
        "machine_failure",
        "twf",
        "hdf",
        "pwf",
        "osf",
        "rnf"
        ]

    for col in target_cols:
        df_features[col] = df_features[col].astype("category")


    cols = incl_cols.copy()
    cols.append(target)
    df_output = df_features[cols]
    
    return df_output

In [ ]:
df = load_data(target = "machine_failure",incl_cols=['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],)
for col in ['air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque']:
    print(f"st.slider(label='{lab_dict[col]}', min_value={int(df[col].min())}, max_value={int(df[col].max())}, value={int(df[col].min()+((df[col].max()-df[col].min())/2))} , step=1 , key='{col}')")

st.slider(label='Air temperature [K]', min_value=295, max_value=304, value=309 , step=1 , key='air_temperature_k')
st.slider(label='Process temperature [K]', min_value=305, max_value=313, value=317 , step=1 , key='process_temperature_k')
st.slider(label='Rotational speed [RPM]', min_value=1168, max_value=2886, value=3745 , step=1 , key='rotational_speed_rpm')
st.slider(label='Torque [Nm]', min_value=3, max_value=76, value=113 , step=1 , key='torque_nm')
st.slider(label='Tool wear [min]', min_value=0, max_value=253, value=379 , step=1 , key='tool_wear_min')
st.slider(label='Power [kW]', min_value=0, max_value=6, value=9 , step=1 , key='power_w')
st.slider(label='Temperature Difference [K]', min_value=7, max_value=12, value=14 , step=1 , key='delta_temp_k')
st.slider(label='Tool Wear [min] x Torque [Nm] ', min_value=0, max_value=16497, value=24745 , step=1 , key='wear_torque')


In [5]:
def split_data(dataframe, stratify, target="machine_failure",seed=42):
    

    y = dataframe[target]
    X = dataframe.drop(target, axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=seed,stratify=stratify )

    return  X_train, X_test, y_train, y_test

In [6]:
@ignore_warnings(category=ConvergenceWarning)

def train_with_aggressive_weights(X_train, y_train, scaler, approach_key, approach_input, categorical_features, hidden_layer_sizes, alpha,learning_rate_init, batch_size, activation, solver):
    
    classes = np.unique(y_train)
    minority_class = classes[np.argmin(np.bincount(y_train))]


    # apply class weights
    if approach_key == "class_weights":
        # Compute and amplify class weights
        weights = compute_class_weight('balanced', classes=classes, y=y_train)
    
        # Amplify minority class weight by 1.5x
        class_weights = dict(zip(classes, weights))
        class_weights[minority_class] *= approach_input
        sample_weights = np.array([class_weights[c] for c in y_train])
    
    numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

    preprocess = ColumnTransformer(transformers=[    
        ("encode_cats", OrdinalEncoder(), categorical_features),
        ("scale", scaler, numerical_features)
        ],remainder="drop")



    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        alpha=alpha,
        batch_size=batch_size,
        learning_rate='adaptive',
        learning_rate_init=learning_rate_init,
        max_iter=1500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42,
        verbose=0
    )


    if approach_key == "class_weights":
        pipeline = Pipeline(steps=[
            ('preprocess', preprocess),
            ('model', mlp),
        ])
    if approach_key == "resampling":
        
    # apply resampling approaches
    
        if approach_input=="ADASYN":
            pipeline = ImbPipeline(steps=[
            ('preprocess', preprocess),
            ("over", ADASYN(random_state=42)),
            ("under",  TomekLinks()),
            ('model', mlp),
            ])
            
        if approach_input=="SMOTETomek":
            pipeline = ImbPipeline(steps=[
            ('preprocess', preprocess),
            ("over_under", SMOTETomek( random_state=42)),
            ('model', mlp),
            ])


    if approach_key == "class_weights":
        pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)
    if approach_key == "resampling":
        pipeline.fit(X_train, y_train)
    

    return pipeline, pipeline.get_params()

In [7]:
def optimize_threshold(model, X_train, y_train, scaler=None):
    """
    Find optimal decision threshold to maximize F1 for minority class
    """
    
    y_proba = model.predict_proba(X_train)[:, 1]
    
    # Try different thresholds
    thresholds = np.arange(0.01, 0.90, 0.01)
    best_f1 = 0
    best_threshold = 0.5
    
    minority_class = np.argmin(np.bincount(y_train))
    
    for threshold in thresholds:
        y_pred_temp = (y_proba >= threshold).astype(int)
        f1 = f1_score(y_train, y_pred_temp, pos_label=minority_class)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    #print(f"Optimal threshold: {best_threshold:.2f} with F1: {best_f1:.4f}")
    return best_threshold

In [8]:
def predict_with_threshold(model, X, threshold=0.5):
    """
    Make predictions using custom threshold
    """
    
    y_proba = model.predict_proba(X)[:, 1]
    
    return (y_proba >= threshold).astype(int)

In [9]:
def report_progress(runs, i, duration, total_duration, f1_result):
    progress = round((i/runs)*100,1)
    if progress == 0:
        feedback = f"process with {runs} runs started at {time.ctime(time.time())}"
    elif progress == 100:
        feedback = f"process with {runs} runs completed in {round(total_duration/60,1)}min at {time.ctime(time.time())}"
    else:
        remaining = round((((100/progress)*total_duration)-total_duration)/60,1)
        feedback = f"progress: {progress}%  |  remaining: {remaining}min   |  elapsed: {round(total_duration/60,1)}min  |   loop duration: {round(duration/60,1)}min    |   best f1: {f1_result}"
    
    return feedback

In [20]:
target = "machine_failure"
categorical_feature = ["type"]
column_selection = {
    "none" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    "rem" : ['type','process_temperature_k','rotational_speed_rpm','tool_wear_min','power_w','delta_temp_k','wear_torque'],

    #"type" : ['air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    "air_temperature_k" : ['type','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    #"process_temperature_k" : ['type','air_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    #"rotational_speed_rpm" : ['type','air_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    #"torque_nm" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','tool_wear_min','power_w','delta_temp_k','wear_torque'],
    #"tool_wear_min" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','power_w','delta_temp_k','wear_torque'],
    #"power_w" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','delta_temp_k','wear_torque'],
    #"delta_temp_k" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','wear_torque'],
    #"wear_torque" : ['type','air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k'],
    #"temp_air_and_delta" : ['type','process_temperature_k','tool_wear_min','power_w','wear_torque'],
    #"wear_torque_air_temp" : ['type','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min','power_w','delta_temp_k'],
    }


model_hidden_layer_sizes= [
                #(10,),
                #(16,),
                #(10, 5),
                (16, 8),
                #(64, 32, 16),
            #    (128, 64, 32),
                #(256, 64, 16),
            #    (256, 128, 64, 32),
                #(256, 64, 32, 16),
                ]
model_alpha = [
                #0.00001, 
            #    0.001, 
                0.01,
            #    1,
                #10,
                ]
model_learning_rate_init = [
                #0.00001, 
            #    0.001,
                0.01,
                #0.1,
                ]
model_batch_size = [
                "auto", 
                16,
            #    64,
            #    256,
                ]
model_activation = [
                'relu', 
            #    'tanh', 
                #'logistic'
                ]
model_solver = [
                'adam', 
                'sgd', 
            #    'lbfgs'
                ]

# try different methods to balance classes
imbalance_approach = {
    "class_weights": [
        #1,
        1.5,
        # 2,
           ], 
    #"resampling": ["SMOTETomek", "ADASYN"]
    }

# try different scalers
scalers = {"power":make_pipeline(PowerTransformer())}#, "robust":make_pipeline(RobustScaler())}#,"standard":make_pipeline(StandardScaler()),"quantile":make_pipeline(QuantileTransformer())}

# define dictionary for result collection
results =  {"excluded_column":[], "imbalanced_approach":[],"imbalanced_input":[],"scaler":[], "learning_rate_init":[],"alpha":[],"batch_size":[],"hidden_layer_sizes":[],"activation":[], "solver":[], "prediction_threshold":[],"duration":[], "f1_score":[]}#,"not null gt":[],"exclude":[],"columns":[]}#, "coefficients":[]}

# calculate number of runs for progress reporting
runs = len(column_selection)*len(imbalance_approach)*len(imbalance_approach["class_weights"])*len(scalers)*len(model_hidden_layer_sizes)*len(model_alpha)*len(model_learning_rate_init)*len(model_batch_size)*len(model_activation)*len(model_solver)
i, total_duration, duration, f1_result=0,0,0,0

for approach_key, approach_item in imbalance_approach.items():
    
    for column_key, cols in column_selection.items():
        
        categorical_feat = categorical_feature
        if categorical_feat[0] not in cols:
            categorical_feat=[]
        df_data = load_data(cols, target)
        
        X_train, X_test, y_train, y_test = split_data(df_data, df_data[target], target=target)
        

        for imbalanced_input in approach_item:
        
            for scaler_key, scaler_value in scalers.items():

                for hidden_layer_sizes in model_hidden_layer_sizes:
                    for alpha in model_alpha:
                        for learning_rate_init in model_learning_rate_init:
                            for batch_size in model_batch_size:
                                for activation in model_activation:
                                    for solver in model_solver:

                                        # report progress and remaining computation time
                                        
                                        print(report_progress(runs, i, duration, total_duration, f1_result))
                                        
                                        i +=1
                                        start_time = time.time()

                                        model, parameters = train_with_aggressive_weights(X_train, y_train, scaler_value,approach_key, imbalanced_input, categorical_feat, hidden_layer_sizes, alpha,learning_rate_init, batch_size, activation, solver)

                                        best_threshold = optimize_threshold(model, X_train, y_train)
                                        y_pred = predict_with_threshold(model,X_test,best_threshold)
                                        end_time = time.time()
                                        duration = round(end_time-start_time,0)
                                        total_duration = total_duration + duration
                                        f1_run = round(f1_score(y_test, y_pred)*100,0)
                                        if f1_run > f1_result:
                                            f1_result = f1_run
                                        # collect result
                                        results['excluded_column'].append(column_key)
                                        results['imbalanced_approach'].append(approach_key)
                                        results['imbalanced_input'].append(imbalanced_input)
                                        results['scaler'].append(scaler_key)
                                        results['learning_rate_init'].append(parameters['model__learning_rate_init'])
                                        results['alpha'].append(parameters['model__alpha'])
                                        results['batch_size'].append(parameters['model__batch_size'])
                                        results['hidden_layer_sizes'].append(parameters['model__hidden_layer_sizes'])
                                        results['activation'].append(parameters['model__activation'])
                                        results['solver'].append(parameters['model__solver'])
                                        results['prediction_threshold'].append(best_threshold)
                                        results['f1_score'].append(f1_run)
                                        results['duration'].append(duration)
            

print(report_progress(runs, i, duration, total_duration, f1_result))
#results.to_csv("results_detailed_run.csv")



process with 12 runs started at Fri Dec  5 12:48:20 2025
progress: 8.3%  |  remaining: 0.2min   |  elapsed: 0.0min  |   loop duration: 0.0min    |   best f1: 46.0
progress: 16.7%  |  remaining: 0.2min   |  elapsed: 0.1min  |   loop duration: 0.0min    |   best f1: 59.0
progress: 25.0%  |  remaining: 0.4min   |  elapsed: 0.1min  |   loop duration: 0.1min    |   best f1: 61.0
progress: 33.3%  |  remaining: 1.0min   |  elapsed: 0.5min  |   loop duration: 0.3min    |   best f1: 71.0
progress: 41.7%  |  remaining: 0.7min   |  elapsed: 0.5min  |   loop duration: 0.0min    |   best f1: 71.0
progress: 50.0%  |  remaining: 0.5min   |  elapsed: 0.5min  |   loop duration: 0.0min    |   best f1: 71.0
progress: 58.3%  |  remaining: 0.5min   |  elapsed: 0.7min  |   loop duration: 0.2min    |   best f1: 71.0
progress: 66.7%  |  remaining: 0.5min   |  elapsed: 1.1min  |   loop duration: 0.3min    |   best f1: 73.0
progress: 75.0%  |  remaining: 0.4min   |  elapsed: 1.1min  |   loop duration: 0.0min   

In [19]:
#results=pd.DataFrame(results)
pd.DataFrame(results).sort_values("f1_score",ascending=False)
#results.to_csv("results_detailed_run.csv")


,excluded_column,imbalanced_approach,imbalanced_input,scaler,learning_rate_init,alpha,batch_size,hidden_layer_sizes,activation,solver,prediction_threshold,duration,f1_score
5,rem,class_weights,1.5,power,0.01,0.01,16,"(16, 8)",relu,sgd,0.85,21.0,73.0
1,none,class_weights,1.5,power,0.01,0.01,16,"(16, 8)",relu,sgd,0.84,21.0,71.0
6,rem,class_weights,1.5,power,0.01,1.00,auto,"(16, 8)",relu,sgd,0.79,2.0,60.0
0,none,class_weights,1.5,power,0.01,0.01,auto,"(16, 8)",relu,sgd,0.89,2.0,59.0
4,rem,class_weights,1.5,power,0.01,0.01,auto,"(16, 8)",relu,sgd,0.82,2.0,59.0
2,none,class_weights,1.5,power,0.01,1.00,auto,"(16, 8)",relu,sgd,0.83,2.0,56.0
7,rem,class_weights,1.5,power,0.01,1.00,16,"(16, 8)",relu,sgd,0.60,20.0,44.0
3,none,class_weights,1.5,power,0.01,1.00,16,"(16, 8)",relu,sgd,0.73,14.0,41.0


In [11]:
# Set options to display all rows and columns
#print(results)
#results=pd.DataFrame(results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 0)

results.sort_values("f1_score",ascending=False)


,excluded_column,imbalanced_approach,imbalanced_input,scaler,learning_rate_init,alpha,batch_size,hidden_layer_sizes,activation,solver,prediction_threshold,duration,f1_score
1567,wear_torque_air_temp,class_weights,1.5,power,0.010,0.010,16,"(16, 8)",relu,sgd,0.81,19.0,77.0
1315,wear_torque_air_temp,class_weights,1,power,0.010,0.001,16,"(16, 8)",relu,sgd,0.82,17.0,75.0
1531,wear_torque_air_temp,class_weights,1.5,power,0.010,0.001,16,"(16, 8)",relu,sgd,0.88,18.0,74.0
1260,none,class_weights,1.5,power,0.001,0.010,16,"(256, 128, 64, 32)",relu,adam,0.89,21.0,73.0
1638,wear_torque_air_temp,class_weights,1.5,power,0.010,0.010,16,"(128, 64, 32)",relu,adam,0.68,26.0,73.0
1138,none,class_weights,1.5,power,0.010,0.010,16,"(16, 8)",tanh,sgd,0.85,17.0,73.0
594,wear_torque_air_temp,resampling,SMOTETomek,power,0.010,0.001,16,"(256, 128, 64, 32)",relu,adam,0.89,79.0,73.0
1213,none,class_weights,1.5,power,0.010,0.010,64,"(128, 64, 32)",relu,sgd,0.89,15.0,72.0
1476,wear_torque_air_temp,class_weights,1,power,0.001,0.010,16,"(256, 128, 64, 32)",relu,adam,0.89,27.0,72.0
996,none,class_weights,1,power,0.010,0.010,64,"(128, 64, 32)",relu,adam,0.88,6.0,72.0


In [12]:
old_res = pd.read_csv("results_loop.csv")
old_res.sort_values("f1_score",ascending=False)

,Unnamed: 0,excluded_column,imbalanced_approach,imbalanced_input,scaler,best_learning_rate_init,best_alpha,best_batch_size,best_hidden_layer_sizes,best_activation,best_solver,prediction_threshold,duration,f1_score
24,24,rotational_speed_rpm,resampling,SMOTE,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.87,30.0,77.0
18,18,process_temperature_k,resampling,SMOTE,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.87,30.0,77.0
73,73,type,class_weights,1.25,robust,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.69,9.0,76.0
79,79,air_temperature_k,class_weights,1.25,robust,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.88,11.0,76.0
78,78,air_temperature_k,class_weights,1.25,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.83,12.0,76.0
22,22,process_temperature_k,resampling,SMOTETomek,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.87,43.0,75.0
28,28,rotational_speed_rpm,resampling,SMOTETomek,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.87,41.0,75.0
94,94,rotational_speed_rpm,class_weights,1.75,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.89,10.0,73.0
101,101,torque_nm,class_weights,1.75,robust,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.88,11.0,73.0
88,88,process_temperature_k,class_weights,1.75,power,0.001,0.001,16,"(128, 64, 32)",relu,adam,0.89,10.0,73.0


In [ ]:
    grid_random_search=True
    
if grid_random_search == True:
        mlp = MLPClassifier(
            learning_rate='adaptive',
            max_iter=1500,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=30,
            random_state=42,
            verbose=False
            )
        param_grid = {
            'model__hidden_layer_sizes': [
                #(10,),
                #(16,),
                #(10, 5),
                #(16, 8),
                #(64, 32, 16),
                (128, 64, 32),
                (256, 64, 16),
                (64, 32, 16, 8),
                #(256, 64, 32, 16),
                ],
            'model__alpha': [
                #0.00001, 
                0.001, 
                0.01,
                #1,
                #10,
                ],
            'model__learning_rate_init': [
                #0.00001, 
                #0.001,
                0.01,
                #0.1,
                ],
            'model__batch_size': [
                #"auto", 
                16,
                64,
                256,
                ],
            'model__activation': [
                'relu', 
                'tanh', 
                #'logistic'
                ],
            'model__solver': [
                'adam', 
                'sgd', 
                'lbfgs'
                ],
        }

f1_scorer = make_scorer(f1_score, pos_label=minority_class)

        if grid_random_search == True:
        model = GridSearchCV(        #RandomizedSearchCV(
            pipeline,
            param_grid,
            cv=StratifiedKFold(n_splits=5),
            scoring=f1_scorer,
            n_jobs=-1,
            verbose=1
        )
    else:

if grid_random_search == True:
        best_model=model.best_estimator_
        best_params = model.best_params_
    else: